# 07 — Validation de l'univers ETF PEA

Validation technique des historiques via yfinance. **Paper trading uniquement : aucun ordre réel.**

In [1]:
import sys
from pathlib import Path
from tempfile import gettempdir

import pandas as pd
import yfinance as yf


def find_project_root(markers=("src", "config")) -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if all((candidate / marker).is_dir() for marker in markers):
            return candidate
    raise RuntimeError("Racine du projet introuvable.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.universe import generate_validation_report, load_universe, validate_universe

CONFIG_PATH = PROJECT_ROOT / "config" / "assets.yaml"
REPORT_PATH = PROJECT_ROOT / "reports" / "universe_validation_report.md"
yf.set_tz_cache_location(str(Path(gettempdir()) / "personal-quant-lab-yfinance"))

## Chargement de l'univers complet

In [2]:
# include_inactive=True conserve les ETF indisponibles dans l'audit.
assets = load_universe(CONFIG_PATH, include_inactive=True)
assert len(assets) == 12
print("Actifs configurés :", len(assets))

Actifs configurés : 12


## Validation yfinance

In [3]:
validation_results = validate_universe(assets, "2015-01-01", "2026-08-03")
validation_frame = pd.DataFrame(validation_results)
validation_frame

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: GOVF.PA"}}}


$GOVF.PA: possibly delisted; no timezone found



1 Failed download:


['GOVF.PA']: possibly delisted; no timezone found


Historique yfinance vide pour GOVF.PA


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CRPE.PA"}}}


$CRPE.PA: possibly delisted; no timezone found



1 Failed download:


['CRPE.PA']: possibly delisted; no timezone found


Historique yfinance vide pour CRPE.PA


,ticker,name,category,universe_active,first_date,last_date,history_months,flags,notes
0,CW8.PA,MSCI World,monde,True,2015-01-02,2026-07-31,139.0,[],
1,ESE.PA,S&P 500 ESG,us_esg,True,2015-01-02,2026-07-31,139.0,[],
2,WPEA.PA,MSCI World PEA synthétique,monde,True,2024-04-02,2026-07-31,28.0,[SHORT_HISTORY],Historique court depuis 2024-04-02.
3,PAEEM.PA,MSCI EM PEA,emergents,True,2019-04-25,2026-07-31,88.0,[],
4,PUST.PA,S&P 500 PEA,us,True,2015-01-02,2026-07-31,139.0,[],
5,PCEU.PA,MSCI Europe,europe,True,2019-04-25,2026-07-31,88.0,[],
6,PAASI.PA,MSCI Pacific ex-Japan PEA,pacifique,True,2019-04-25,2026-07-31,88.0,[],"Remplace PXPM.PA, indisponible sur yfinance."
7,GWT.PA,MSCI World Small Cap,small_caps,True,2024-04-25,2026-07-31,28.0,[SHORT_HISTORY],Historique court depuis 2024-04-25.
8,OBLI.PA,Obligations EUR agrégat,obligataire,True,2018-12-17,2026-07-31,92.0,[],
9,GOVF.PA,EuroMTS All-Maturity,obligations_souveraines,False,None,None,NaN,[UNAVAILABLE],UNAVAILABLE sur yfinance au 2026-08-03. Donnée...


## Génération du rapport Markdown

In [4]:
# Écriture déterministe : rejouer cette cellule remplace proprement le rapport.
generate_validation_report(validation_results, REPORT_PATH)
assert REPORT_PATH.exists() and REPORT_PATH.stat().st_size > 0
print("Rapport généré :", REPORT_PATH.relative_to(PROJECT_ROOT))

Rapport généré : reports\universe_validation_report.md
